# seu_suite — Colab 실행 노트북
우주 방사선 비트플립 시뮬레이션(2026-09-11 수정본)을 Google Colab에서 돌리는 노트북입니다.

**주의**
- 런타임 유형은 **CPU**로 둡니다 (GPU 불필요, 코드가 CPU 전용).
- 설치는 필요 없습니다 (torch·torchvision·pandas·matplotlib 기본 제공).
- 세션이 끊기면 `/content`가 지워지므로 마지막 셀로 결과를 Drive에 저장하세요.

위에서부터 순서대로 실행합니다. 각 실험 셀은 필요한 것만 골라 실행해도 됩니다.

## 1. 코드 올리기 — 방법 A: zip 직접 업로드 (`seu_suite_v2.zip` 선택)

In [ ]:
from google.colab import files
up = files.upload()   # 파일 선택 창에서 seu_suite_v2.zip 선택
!unzip -o -q seu_suite_v2.zip -d /content
%cd /content/seu_suite
!ls

### (방법 B) Google Drive에 zip을 넣어 두었다면 이 셀을 대신 실행 (`내 드라이브/seu/seu_suite_v2.zip`)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -o -q "/content/drive/MyDrive/seu/seu_suite_v2.zip" -d /content
# %cd /content/seu_suite
# !ls

## 2. 자가진단 — 첫 줄에 `[self-test] bit-flip OK`가 나와야 합니다

In [ ]:
!python -c "import torch, torchvision, matplotlib; print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| matplotlib', matplotlib.__version__)"
!python simulate.py --model toy50 --trials 1

## 3-1. E1 용량-반응: 확률 p를 바꿔가며 (mnist_linear, 방어 없음 vs clip)
처음 실행 시 MNIST(12MB)를 내려받고 깨끗한 모델을 학습해 `results/ckpts/`에 저장합니다.

In [ ]:
for p in [0.0001, 0.001, 0.003, 0.01, 0.03]:
    !python simulate.py --model mnist_linear --mode infer --rad stress --p {p} --defense none --trials 10
    !python simulate.py --model mnist_linear --mode infer --rad stress --p {p} --defense clip --trials 10

## 3-2. E2 비트 하나의 운명: 정확히 한 번, 지정한 비트만

In [ ]:
for b in [31, 30, 29, 27, 23, 22, 10, 0]:
    !python simulate.py --model mnist_linear --mode infer --rad stress --single-hit --allowed-bits {b} --trials 30

## 3-3. E3 궤도 임무: 며칠이면 죽나 (leo_saa, T는 하루의 정수배)

In [ ]:
for T in [1, 7, 30, 90]:
    !python simulate.py --model mnist_linear --mode infer --rad leo_saa --T {T} --defense none --trials 10
    !python simulate.py --model mnist_linear --mode infer --rad leo_saa --T {T} --defense clip --trials 10

## 3-4. E5 방어 대결 (toy50, p=5%) — `ensemble`(평균) vs `ensemble_median`(중앙값) 차이에 주목

In [ ]:
for d in ["none", "clip", "tmr", "ensemble", "ensemble_median", "parity"]:
    !python simulate.py --model toy50 --mode infer --rad stress --p 0.05 --defense {d} --trials 20

## 3-5. E4 학습 중 공격 (mnist_linear 1 epoch)

In [ ]:
for d in ["none", "clip"]:
    !python simulate.py --model mnist_linear --mode train --rad stress --p 0.01 --defense {d} --trials 5 --epochs 1

## 3-6. (선택) 논문 전체 표 한 번에 — 20~30분, CIFAR-10 170MB + ResNet 가중치 45MB 다운로드
브라우저 탭을 닫지 마세요.

In [ ]:
# !python simulate.py --preset paper_sweep

## 4. 결과 요약표 — 평균 대신 중앙값·생존률로 보기

In [ ]:
import glob, sys, pandas as pd
sys.path.insert(0, "/content/seu_suite")
from simulate import make_summary

files_ = glob.glob("results/single_*.csv")
df = pd.concat([pd.read_csv(f) for f in files_], ignore_index=True)
summ = make_summary(df)
cols = ["model","mode","rad","defense","allowed_bits","single_hit","p","T_days","n","median_acc","survival_rate","crash_rate","mean_k"]
pd.set_option("display.width", 200)
summ[cols]

## 4-1. 산점도 (13·14주차 방식): 확률 vs 중앙값 정확도

In [ ]:
import matplotlib.pyplot as plt
s = summ[(summ.model=="mnist_linear") & (summ["mode"]=="infer") & (summ.rad=="stress") & (summ.single_hit==0)]
plt.figure(figsize=(8,5))
for d, g in s.groupby("defense"):
    g = g.sort_values("p")
    plt.plot(g["p"]*100, g["median_acc"], "o-", label=d)
plt.xscale("log"); plt.xlabel("Bit-flip probability (%)"); plt.ylabel("Median accuracy (%)")
plt.ylim(0, 100); plt.grid(alpha=.3); plt.legend(); plt.title("E1 dose-response (mnist_linear)")
plt.show()

## 4-2. 비트 지도 (E2): 비트 번호별 생존률

In [ ]:
b = summ[(summ.model=="mnist_linear") & (summ.single_hit==1)].copy()
if len(b):
    b["bit"] = b["allowed_bits"].astype(int)
    b = b.sort_values("bit")
    colors = ["#245EA6" if x==31 else ("#C94A24" if 23<=x<=30 else "#8A97A8") for x in b["bit"]]
    plt.figure(figsize=(9,4))
    plt.bar(b["bit"].astype(str), b["survival_rate"], color=colors)
    plt.ylabel("Survival rate (acc >= 0.9 clean)"); plt.xlabel("Flipped bit (blue=sign, red=exponent, grey=mantissa)")
    plt.title("E2 one-bit criticality (mnist_linear)"); plt.ylim(0,1.05); plt.grid(axis="y", alpha=.3); plt.show()
else:
    print("3-2 셀을 먼저 실행하세요.")

## 5. 결과 저장 — 반드시 실행 (Drive 복사)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/seu/results_$(date +%Y%m%d_%H%M)"
!cp -r results/* "/content/drive/MyDrive/seu/results_$(date +%Y%m%d_%H%M)/"
!ls "/content/drive/MyDrive/seu/" 

### 또는 zip으로 내려받기

In [ ]:
!zip -r -q results.zip results
from google.colab import files
files.download("results.zip")

## 6. (심화) 함수를 직접 불러 쓰기 — 코드 리뷰 문서의 단계와 같은 이름
9주차 실습 2-7의 논문 버전: 깨끗한 모델의 가중치 하나에서 30번 비트만 뒤집고 채점

In [ ]:
import sys, copy, numpy as np, torch
sys.path.insert(0, "/content/seu_suite")
import radiation as R, defenses as D, engine as E, models as M

data = M.load_data_for("mnist_linear", 8000, 2000)
fac = lambda: M.build_model("mnist_linear")
state, clean_acc = E.load_or_train_clean(model_factory=fac, data=data, model_name="mnist_linear",
                                         epochs=2, batch_size=128, lr=0.05, momentum=0.9, ckpt_dir="results/ckpts")
print("clean acc:", clean_acc)

for bit in [30, 23, 0]:
    accs = []
    for seed in range(10):
        m = fac(); m.load_state_dict(copy.deepcopy(state))
        params = R.collect_weight_params(m); cum, n_w = R.weight_bank_meta(params)
        idx = int(np.random.default_rng(seed).integers(0, n_w))
        R.xor_one_bit(params, cum, idx, bit)
        a = E.evaluate(m, data.x_test, data.y_test, "mnist_linear")
        accs.append(a if a is not None else float("nan"))
    print(f"bit {bit:>2}: {np.round(accs, 1)}")